# ทดสอบโมเดลกับเสียงจากไมค์จริง (INMP441)

1. เลือก run ที่จะทดสอบ (จาก `runs/<name>/<time>/`)
2. เลือกไฟล์ใน `data/raw/inmp441/records/` แล้วบอกว่าไฟล์นั้นเป็นเสียงอะไร (`EXPECTED`) หรือระบุช่วงเวลาจริง (`TRUTH`)
3. โมเดลเลื่อนหน้าต่าง 1 s ทีละ 0.5 s (ซ้อนกัน 0.5 s) แล้วทายทุกหน้าต่าง
4. ดูตาราง กราฟ และฟังหน้าต่างที่สนใจ

Hamming taper อยู่ใน `model.feat` (ตาม `features.taper` ใน config) จึงใช้แบบเดียวกับตอนเทรนเสมอ ไม่ต้องคูณเองใน notebook

**ระวัง:** ไฟล์ INMP441 ที่อยู่ใน `data/clips.csv` ถูกใช้เทรน/val/test ไปแล้ว คอลัมน์ `used_in` ในตารางไฟล์บอกไว้
ถ้าจะวัดผลจริงให้ใช้ไฟล์ที่อัดใหม่และยังไม่ได้รัน `make process`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Audio, display

from coughml.infer import list_runs, load_audio, load_run, sliding_predict

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
runs = list_runs(ROOT / "runs")
runs

## 1. เลือก run

In [ ]:
# default: run ที่ test macro F1 สูงสุด หรือใส่ path เอง เช่น ROOT / "runs/cnn/20261112-173504"
RUN = Path(runs.sort_values("test_macro_f1").iloc[-1]["path"])

model, cfg, classes = load_run(RUN, DEVICE)
SR = cfg["features"]["sample_rate"]
print(RUN.relative_to(ROOT))
print("classes:", classes, "| model:", cfg["model"]["name"], "| taper:", cfg["features"]["taper"])

## 2. เลือกไฟล์และบอกว่าเป็นเสียงอะไร

In [ ]:
REC = ROOT / "data" / "raw" / "inmp441" / "records"
files = pd.DataFrame({"file": sorted(str(p.relative_to(REC)) for p in REC.rglob("*.wav"))})
labels_csv = REC / "labels.csv"
if labels_csv.exists():
    lab = pd.read_csv(labels_csv, encoding="utf-8-sig")[["file", "label", "seconds"]]
    files = files.merge(lab, on="file", how="left")
clips_csv = ROOT / "data" / "clips.csv"
if clips_csv.exists():
    c = pd.read_csv(clips_csv)
    c = c[c.dataset == "inmp441"]
    used = c.groupby(c.interim_path.str.replace("inmp441/records/", "", regex=False)).split.agg(
        lambda s: ",".join(sorted(set(s))))
    files["used_in"] = files.file.map(used).fillna("-")
files

In [ ]:
FILE = files.file.iloc[0]          # หรือพิมพ์เอง เช่น "cough/fococus_room_20261112_154810.wav"
EXPECTED = "cough"                 # ทั้งไฟล์เป็นเสียงอะไร: cough / speech / background (None = ไม่เทียบ)

# ถ้ารู้ช่วงเวลาจริง ใส่เป็น (start_s, end_s, label) แล้วหน้าต่างที่ไม่ทับช่วงไหนเลยจะถือเป็น OUTSIDE
TRUTH = []                         # เช่น [(2.1, 2.6, "cough"), (5.0, 5.4, "cough")]
OUTSIDE = "background"
MIN_OVERLAP_S = 0.2                # หน้าต่างต้องทับช่วงจริงอย่างน้อยเท่านี้ถึงนับเป็น label นั้น

WIN_S, HOP_S = 1.0, 0.5
COUGH_THRESHOLD = 0.5              # p_cough >= ค่านี้ ถือว่าเจอเสียงไอ

## 3. Sliding window

In [ ]:
y = load_audio(REC / FILE, SR)
df = sliding_predict(model, y, classes, SR, WIN_S, HOP_S)
df["is_cough"] = df["p_cough"] >= COUGH_THRESHOLD


def truth_of(start, end):
    best, best_ov = OUTSIDE, 0.0
    for s, e, lb in TRUTH:
        ov = min(end, e) - max(start, s)
        if ov >= MIN_OVERLAP_S and ov > best_ov:
            best, best_ov = lb, ov
    return best


if TRUTH:
    df["truth"] = [truth_of(s, e) for s, e in zip(df.start_s, df.end_s)]
elif EXPECTED:
    df["truth"] = EXPECTED
if "truth" in df:
    df["correct"] = df["pred"] == df["truth"]

print(f"{FILE}: {len(y) / SR:.2f} s, {len(df)} windows")
pd.set_option("display.max_rows", 200)
df

In [ ]:
print("predicted windows:", df.pred.value_counts().to_dict())
print("cough windows (p_cough >= %.2f): %d" % (COUGH_THRESHOLD, df.is_cough.sum()))
if "truth" in df:
    print(f"window accuracy: {df.correct.mean():.3f}")
    for lb in classes:
        t = df[df.truth == lb]
        if len(t):
            print(f"  recall {lb:10s} {(t.pred == lb).mean():.3f}  ({len(t)} windows)")
    display(pd.crosstab(df.truth, df.pred, rownames=["truth"], colnames=["pred"]))

## 4. กราฟ

In [ ]:
import torchaudio

t = np.arange(len(y)) / SR
colors = {"cough": "tab:red", "speech": "tab:blue", "background": "tab:gray"}
fig, ax = plt.subplots(3, 1, figsize=(14, 9), sharex=True, gridspec_kw={"height_ratios": [1.2, 1.2, 1]})

ax[0].plot(t, y, lw=0.4, color="black")
for _, r in df[df.is_cough].iterrows():
    ax[0].axvspan(r.start_s, r.end_s, color="tab:red", alpha=0.12)
for s, e, lb in TRUTH:
    ax[0].axvspan(s, e, ymin=0.92, ymax=1.0, color=colors.get(lb, "green"), alpha=0.9)
ax[0].set_title(f"{FILE}  (red = window predicted cough; top bar = TRUTH)")
ax[0].set_ylabel("amplitude")

mel = torchaudio.transforms.MelSpectrogram(SR, n_fft=cfg["features"]["n_fft"],
                                           hop_length=cfg["features"]["hop_length"],
                                           n_mels=cfg["features"]["n_mels"])(torch.from_numpy(y))
ax[1].imshow(torch.log(mel + 1e-6).numpy(), origin="lower", aspect="auto", extent=[0, len(y) / SR, 0, mel.shape[0]])
ax[1].set_ylabel("mel bin")

for c in classes:
    ax[2].plot(df.mid_s, df[f"p_{c}"], marker="o", ms=3, label=c, color=colors.get(c))
ax[2].axhline(COUGH_THRESHOLD, ls="--", color="tab:red", lw=0.8)
ax[2].set_ylim(0, 1.02)
ax[2].set_ylabel("probability (window centre)")
ax[2].set_xlabel("seconds")
ax[2].legend(loc="upper right")
plt.tight_layout()
plt.show()

## 5. ฟังเสียง

In [ ]:
display(Audio(y, rate=SR))  # ทั้งไฟล์

In [ ]:
# หน้าต่างที่สนใจ: default = p_cough สูงสุด หรือเปลี่ยน i เป็นเลขแถวในตาราง
i = int(df.p_cough.idxmax())
r = df.loc[i]
print(f"window {i}: {r.start_s:.2f}-{r.end_s:.2f} s  pred {r.pred}  " +
      "  ".join(f"p_{c} {r[f'p_{c}']:.2f}" for c in classes))
display(Audio(y[int(r.start_s * SR):int(r.end_s * SR)], rate=SR))